# 2026 实验7 调用Renet模型参数搭建CNN分类网络

## 读取数据

In [1]:
import os
from skimage.io import imread
import numpy as np
from skimage.transform import resize

def read_UC(path):
    # 读取文件夹内所有子文件夹的名称
    subfolders = [name for name in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, name))]

    # 初始化变量
    X = []
    Y = []

    # 遍历每个子文件夹
    for index, subfolder in enumerate(subfolders):
        # 获取子文件夹的路径和类别编号
        subfolder_path = os.path.join(folder_path, subfolder)
        label = index
        
        # 遍历子文件夹内的所有tif图像
        for filename in os.listdir(subfolder_path):
            if filename.endswith('.tif'):
                # 读取图像数据
                image_path = os.path.join(subfolder_path, filename)
                image = imread(image_path)
                image=resize(image,(64,64,3))#为了加快训练，减小图像尺寸
                image = np.transpose(image,(2,0,1)) 
                # 将图像数据添加到X变量中
                X.append(image)
                
                # 将类别编号添加到Y变量中
                Y.append(label)

    # 将X和Y转换为NumPy数组
    X = np.array(X)
    Y = np.array(Y)

    # 打印数据维度
    print('图像数据维度:', X.shape)
    print('标签数据维度:', Y.shape)

    return X,Y

In [4]:
import torch
from torch import nn
import numpy as np

# 文件夹路径
folder_path = '/Users/mf/UCMerced_LandUse/train'

# 读取训练集和测试集数据
[train_img, train_lb] = read_UC('/Users/mf/UCMerced_LandUse/train')
[test_img, test_lb] = read_UC('/Users/mf/UCMerced_LandUse/validation')


# 将所有数据归一化到0-1之间
train_img =train_img/255.
test_img  =test_img/255.

# 对标签进行热编码
one_hot_train_lb = np.eye(9)[train_lb]
one_hot_test_lb= np.eye(9)[test_lb]

# 打印查看数据集格式
print('训练集图像格式为:', train_img.shape, '训练集标签格式为:', train_lb.shape,'热编码训练集标签格式为:', one_hot_train_lb.shape)
print('测试集图像格式为:', test_img.shape, '测试集标签格式为:', test_lb.shape,'热编码测试集标签格式为:', one_hot_test_lb.shape)

图像数据维度: (720, 3, 64, 64)
标签数据维度: (720,)
图像数据维度: (720, 3, 64, 64)
标签数据维度: (720,)
训练集图像格式为: (720, 3, 64, 64) 训练集标签格式为: (720,) 热编码训练集标签格式为: (720, 9)
测试集图像格式为: (720, 3, 64, 64) 测试集标签格式为: (720,) 热编码测试集标签格式为: (720, 9)


## 使用部分resnet18卷积层——搭建自己的网络

###   要求1：使用Resnet18中的(conv1)、(bn1)、(relu)、(maxpool)层+(layer1)块+(layer2)块中的(conv1)、(bn1)、(relu)+两个自己的全连接层，组成分类网络，成功训练/测试


In [5]:
import torchvision.models as models
models.resnet18(weights='IMAGENET1K_V1')

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [6]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        # 补充完整
        self.w1 =nn.Linear(   ,   )
        self.w2 =nn.Linear(   ,   )
        self.BN3=nn.BatchNorm1d(   )
        self.relu=nn.ReLU()
        # 补充结束
        self.Resnet=models.resnet18(weights='IMAGENET1K_V1')

    def forward(self, x):
        # 补充完整




        # 补充结束
    
        return x
model = NeuralNetwork()

## 训练网络

In [7]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

learning_rate = 5e-3
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

batch_size = 20
epochs = 5
batch_num=int(train_img.shape[0]/batch_size)
size = len(train_img)

model.train()
for t in range(epochs):
    
    correct=0.
    train_mean_loss=0.

    for batch in range(batch_num):
        X=train_img[batch*batch_size:(batch+1)*batch_size,]
        y=one_hot_train_lb[batch*batch_size:(batch+1)*batch_size,:]

        X=torch.tensor(X, dtype=torch.float32)
        y=torch.tensor(y, dtype=torch.float32)
        
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        correct += (pred.argmax(1) == y.argmax(1)).type(torch.float).mean().item()
        train_mean_loss+= loss.item()

    train_mean_loss /= batch_num
    correct /= batch_num

    print(f" Epoch:{t}, loss: {train_mean_loss:>8f},  Accuracy: {(100*correct):>0.1f}%")
torch.save(model,"resnet_classification.pth")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (20x8192 and 1000x100)

## 测试CNN在测试数据集上的Performance

In [47]:
torch.load("resnet_classification.pth")

model.eval()
test_loss, correct = 0, 0
with torch.no_grad():
        X=torch.tensor(test_img, dtype=torch.float32)
        y=torch.tensor(one_hot_test_lb, dtype=torch.float32)
        pred = model(X)
        
        test_loss = np.mean(loss_fn(pred, y).item())
        correct = (pred.argmax(1) == y.argmax(1)).type(torch.float).mean().item()
print(f"Test Accuracy: {(100*correct):>0.1f}%, Test Avg loss: {test_loss:>8f} \n")

Test Accuracy: 89.7%, Test Avg loss: 1.146060 



### 要求2： 对比与3层卷积CNN网络的分类性能差异。

### 要求3： 将数据读取部分，修改为使用Datasets和Dataloader。